# Activity: Foundational Statistical Outlier Detection

## Learning Objectives

By the end of this activity, you will be able to:

1. Identify outliers using visualization techniques (histograms, box plots, violin plots)
2. Implement and apply Tukey's IQR method for outlier detection
3. Detect outliers using Z-score and Modified Z-score methods
4. Understand the limitations of univariate outlier detection
5. Apply multivariate outlier detection using Mahalanobis Distance and Isolation Forest
6. Compare and contrast different outlier detection methods

## Dataset

We'll use a weight-height dataset containing measurements from individuals. This dataset is ideal for learning outlier detection because:
- It has clear univariate outliers (extreme values in single variables)
- It demonstrates multivariate outliers (unusual combinations of values)
- The data is intuitive and easy to interpret

## Setup and Imports

In [ ]:
# Check library versions
import matplotlib 
import pandas as pd
import scipy 
import statsmodels

print(f'''
matplotlib -> {matplotlib.__version__}
pandas -> {pd.__version__}   
scipy -> {scipy.__version__}
statsmodels -> {statsmodels.__version__}
''')

In [ ]:
# Main imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import scipy.stats as stats
from sklearn.ensemble import IsolationForest
from scipy.spatial.distance import mahalanobis

# Set visualization defaults
plt.rcParams["figure.figsize"] = [12, 5]
sns.set_style("whitegrid")

## Load and Explore Data

In [ ]:
# Load the weight-height dataset
file = Path("../data/weight-height.csv")
wh = pd.read_csv(file)

# Display basic information
print(f"Dataset shape: {wh.shape}")
print(f"\nFirst few rows:")
wh.head()

In [ ]:
# Summary statistics
wh.describe()

In [ ]:
# Check distribution of Gender
wh['Gender'].value_counts()

---

# Recipe 1: Detecting Outliers using Visualization

## Learning Objectives
- Understand how different visualizations reveal outliers
- Learn when to use histograms vs. box plots vs. violin plots
- Recognize the subjective nature of visual outlier detection

## Key Concepts

Visual inspection is often the first step in outlier detection. Different plots reveal different aspects:
- **Histograms**: Show distribution shape and extreme values
- **Box plots**: Highlight statistical outliers using quartiles
- **Violin plots**: Combine distribution shape with quartile information

## 1.1 Histogram Visualization

In [ ]:
# Simple histogram of both variables
sns.histplot(wh)
plt.title('Distribution of Height and Weight')
plt.show()

In [ ]:
# Separate histograms with better layout
g = sns.displot(wh, kind='hist', height=5, aspect=2)
g.fig.suptitle('Distribution of Height and Weight', y=1.02)
plt.show()

## 1.2 Box Plot Visualization

Box plots show:
- **Box**: Interquartile range (IQR, 25th to 75th percentile)
- **Line in box**: Median
- **Whiskers**: Typically extend to 1.5 × IQR
- **Points beyond whiskers**: Outliers

In [ ]:
# Box plot for both variables
sns.boxplot(wh, orient='h', whis=1.5)
plt.title('Box Plot of Height and Weight (1.5 × IQR)')
plt.show()

In [ ]:
# Individual box plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(wh['Height'], orient='h', whis=1.5, ax=axes[0])
axes[0].set_title('Height Distribution')

sns.boxplot(wh['Weight'], orient='h', whis=1.5, ax=axes[1])
axes[1].set_title('Weight Distribution')

plt.tight_layout()
plt.show()

## 1.3 Violin Plot Visualization

Violin plots combine the box plot with a kernel density estimation.

In [ ]:
# Violin plot showing quartiles
sns.violinplot(wh, inner='quartile', orient='h')
plt.title('Violin Plot of Height and Weight')
plt.show()

### Exercise 1.1: Visual Analysis

Based on the visualizations above, answer these questions:

1. Which variable (Height or Weight) appears to have more outliers?
2. Do the outliers appear symmetric (both high and low) or skewed to one side?
3. What are the advantages and disadvantages of visual outlier detection?

**Your answers here:**
- Question 1: _[Your answer]_
- Question 2: _[Your answer]_
- Question 3: _[Your answer]_

---

# Recipe 2: Detecting Outliers using Tukey's Method (IQR)

## Learning Objectives
- Understand Tukey's fence method for outlier detection
- Implement the IQR calculation and boundary determination
- Learn how to adjust sensitivity using the k parameter

## Key Concepts

**Tukey's Method** defines outliers as points that fall outside these fences:
- **Lower fence**: Q1 - k × IQR
- **Upper fence**: Q3 + k × IQR

Where:
- Q1 = 25th percentile, Q3 = 75th percentile
- IQR = Q3 - Q1 (Interquartile Range)
- k = sensitivity parameter (commonly 1.5 for outliers, 3.0 for extreme outliers)

## Understanding Percentiles and Quantiles

In [ ]:
# Examine key percentiles
percentiles = [0, 5, 10, 25, 50, 75, 90, 95, 100]
weight_percentiles = np.percentile(wh['Weight'], percentiles)

print("Weight Percentiles:")
for p, v in zip(percentiles, weight_percentiles):
    print(f"  {p:3d}th percentile: {v:.2f}")

## 🎯 Your Task: Implement Tukey's Method

Complete the `iqr_outliers` function below. You need to:
1. Calculate Q1 (25th percentile) and Q3 (75th percentile)
2. Calculate the IQR
3. Calculate lower and upper fences
4. Return data points outside the fences

In [ ]:
def iqr_outliers(data, k=1.5):
    """
    Detect outliers using Tukey's method with customizable fence multiplier.
    
    Parameters:
    -----------
    data : pandas.Series or numpy.array
        The data to analyze for outliers
    k : float, default=1.5
        The fence multiplier (1.5 for outliers, 3.0 for extreme outliers)
    
    Returns:
    --------
    pandas.Series or numpy.array
        Data points identified as outliers
    """
    # TODO: Calculate Q1 and Q3 using np.percentile
    # Hint: Use percentiles [25, 75]
    q1, q3 = # YOUR CODE HERE
    
    # TODO: Calculate the IQR
    IQR = # YOUR CODE HERE
    
    # TODO: Calculate lower and upper fences
    lower_fence = # YOUR CODE HERE
    upper_fence = # YOUR CODE HERE
    
    # TODO: Return data points outside the fences
    # Hint: Use boolean indexing with | (or) operator
    return # YOUR CODE HERE

## Test Your Implementation

In [ ]:
# Test on Height with k=1.5
print("Testing IQR method on Height:")
sns.boxplot(wh['Height'], orient='h', whis=1.5)
plt.title('Height Box Plot (k=1.5)')
plt.show()

height_outliers = iqr_outliers(wh['Height'], k=1.5)
print(f"\nNumber of Height outliers detected: {len(height_outliers)}")
print(f"Outlier values: {height_outliers.values[:10]}...")  # Show first 10

In [ ]:
# Test on Weight with k=1.5
print("Testing IQR method on Weight:")
sns.boxplot(wh['Weight'], orient='h', whis=1.5)
plt.title('Weight Box Plot (k=1.5)')
plt.show()

weight_outliers = iqr_outliers(wh['Weight'], k=1.5)
print(f"\nNumber of Weight outliers detected: {len(weight_outliers)}")
print(f"Outlier values: {weight_outliers.values[:10]}...")  # Show first 10

### Exercise 2.1: Sensitivity Analysis

Test how the number of detected outliers changes with different k values:

In [ ]:
# TODO: Test with k=1.0, 1.5, 2.0, 3.0
# Count outliers for each k value for Height
k_values = [1.0, 1.5, 2.0, 3.0]

print("Sensitivity Analysis - Height:")
for k in k_values:
    outliers = iqr_outliers(wh['Height'], k=k)
    print(f"k={k}: {len(outliers)} outliers detected")

**Question:** What happens to the number of outliers as k increases? Why?

**Your answer:** _[Your answer here]_

---

# Recipe 3: Detecting Outliers using Z-Scores

## Learning Objectives
- Understand the Z-score method and its assumptions
- Implement Z-score calculation for outlier detection
- Learn when Z-score method is appropriate (normal distributions)

## Key Concepts

**Z-score** measures how many standard deviations a point is from the mean:

$$z = \frac{x - \mu}{\sigma}$$

Where:
- x = data point
- μ = mean
- σ = standard deviation

**Common thresholds:**
- |z| > 2: ~5% of data (95% within)
- |z| > 3: ~0.3% of data (99.7% within)

**Assumption:** Data should be approximately normally distributed

## 🎯 Your Task: Implement Z-Score Detection

Complete the `zscore` function below:

In [ ]:
def zscore(df, threshold=3):
    """
    Detect outliers using z-score method with customizable threshold.
    
    Parameters:
    -----------
    df : pandas.Series
        Data to analyze for outliers
    threshold : float, default=3
        The threshold in standard deviations (typically 2-3)
    
    Returns:
    --------
    tuple: (outliers, transformed)
        - outliers: DataFrame containing outlier points with their z-scores
        - transformed: Full DataFrame with z-scores column added
    """
    data = df.copy()
    
    # TODO: Calculate z-score for each data point
    # Formula: (value - mean) / standard_deviation
    # Hint: Use data.mean() and data.std()
    data['zscore'] = # YOUR CODE HERE
    
    # TODO: Identify outliers where |z-score| > threshold
    # Hint: Use boolean indexing with | (or) operator
    outliers = # YOUR CODE HERE
    
    return outliers, data

## Helper Function for Visualization

This function is provided to help you visualize z-scores:

In [ ]:
def plot_zscore(data_series, d=3, title='Standardized Data with Outlier Thresholds'):
    """
    Plot the standardized z-scores with threshold lines.
    
    Parameters:
    -----------
    data_series : pandas.Series
        Series containing z-scores
    d : float, default=3
        Threshold in standard deviations
    title : str
        Plot title
    """
    plt.figure(figsize=(12, 5))
    plt.plot(data_series.index, data_series.values, 'k^', markersize=4, alpha=0.6)
    
    plt.axhline(y=d, color='r', linestyle='--', label=f'+{d} SD', linewidth=2)
    plt.axhline(y=-d, color='r', linestyle='--', label=f'-{d} SD', linewidth=2)
    plt.axhline(y=0, color='gray', linestyle='-', alpha=0.3, linewidth=1)
    
    # Highlight outliers
    outliers = data_series[abs(data_series) > d]
    if not outliers.empty:
        plt.plot(outliers.index, outliers.values, 'ro', markersize=8, label='Outliers', alpha=0.7)
    
    plt.ylabel('Z-score')
    plt.xlabel('Data Point Index')
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

## Test Your Implementation

In [ ]:
# Apply z-score method to Height with threshold=3
outliers, transformed = zscore(wh['Height'], threshold=3)

print(f"Number of outliers detected: {len(outliers)}")
print(f"\nOutlier statistics:")
print(outliers.describe())

In [ ]:
# Visualize the z-scores
plot_zscore(transformed['zscore'], d=3.0, title='Height Z-Scores (threshold=3)')

In [ ]:
# Check distribution of z-scores
transformed['zscore'].hist(bins=50, edgecolor='black')
plt.axvline(x=3, color='r', linestyle='--', label='Threshold=3')
plt.axvline(x=-3, color='r', linestyle='--')
plt.xlabel('Z-score')
plt.ylabel('Frequency')
plt.title('Distribution of Z-Scores')
plt.legend()
plt.show()

### Exercise 3.1: Compare Thresholds

In [ ]:
# TODO: Compare outlier detection with thresholds 2, 2.5, and 3
thresholds = [2, 2.5, 3]

print("Threshold Comparison - Height:")
for t in thresholds:
    outliers, _ = zscore(wh['Height'], threshold=t)
    print(f"Threshold={t}: {len(outliers)} outliers detected")

### Exercise 3.2: Test Normality Assumption

The z-score method assumes normal distribution. Let's test this:

In [ ]:
from statsmodels.stats.diagnostic import kstest_normal

# Kolmogorov-Smirnov test for normality
def test_normal(data):
    result = kstest_normal(data)
    print(f"KS Statistic: {result[0]:.4f}")
    print(f"P-value: {result[1]:.4f}")
    if result[1] > 0.05:
        print("Conclusion: Data appears normally distributed (p > 0.05)")
    else:
        print("Conclusion: Data does NOT appear normally distributed (p ≤ 0.05)")

print("Normality Test for Height:")
test_normal(wh['Height'])

**Question:** Is the Height data normally distributed? What does this mean for using Z-scores?

**Your answer:** _[Your answer here]_

---

# Recipe 4: Detecting Outliers using Modified Z-Score

## Learning Objectives
- Understand the limitations of standard z-score for non-normal data
- Implement the Modified Z-score using median and MAD
- Compare robustness of Modified Z-score vs. standard Z-score

## Key Concepts

**Modified Z-score** is more robust to outliers because it uses:
- **Median** instead of mean (not affected by extreme values)
- **MAD (Median Absolute Deviation)** instead of standard deviation

$$M_i = \frac{0.6745(x_i - \text{median})}{\text{MAD}}$$

Where:
- MAD = median(|x - median(x)|)
- 0.6745 ≈ Φ⁻¹(0.75) makes MAD comparable to standard deviation

**Threshold:** Typically |M| > 3.5 or |M| > 2.5

## Understanding the Scaling Factor

The 0.6745 factor comes from the standard normal distribution's 75th percentile:

In [ ]:
# The scaling factor links MAD to standard deviation
scaling_factor = stats.norm.ppf(0.75)
print(f"Scaling factor (75th percentile of standard normal): {scaling_factor:.4f}")
print(f"\nThis makes MAD comparable to standard deviation for normal distributions")

## 🎯 Your Task: Implement Modified Z-Score

Complete the `modified_zscore` function below:

In [ ]:
def modified_zscore(df, threshold=3.5):
    """
    Detect outliers using modified z-score method with customizable threshold.
    
    Parameters:
    -----------
    df : pandas.Series
        Data to analyze for outliers
    threshold : float, default=3.5
        The threshold for modified z-scores (typically 2.5-3.5)
    
    Returns:
    --------
    tuple: (outliers, transformed)
        - outliers: DataFrame containing outlier points with modified z-scores
        - transformed: Full DataFrame with modified z-scores column added
    """
    data = df.copy()
    
    # TODO: Calculate median of the data
    median = # YOUR CODE HERE
    
    # TODO: Calculate MAD (Median Absolute Deviation)
    # MAD = median(|x - median(x)|)
    # Hint: Use np.median() and np.abs()
    MAD = # YOUR CODE HERE
    
    # TODO: Calculate scaling factor (75th percentile of standard normal)
    # Hint: Use stats.norm.ppf(0.75)
    s = # YOUR CODE HERE
    
    # TODO: Calculate modified z-score
    # Formula: s * (x - median) / MAD
    data['m_zscore'] = # YOUR CODE HERE
    
    # TODO: Identify outliers where |modified z-score| > threshold
    outliers = # YOUR CODE HERE
    
    return outliers, data

## Test Your Implementation

In [ ]:
# Apply modified z-score method with threshold=2.5
outliers_mod, transformed_mod = modified_zscore(wh['Height'], threshold=2.5)

print(f"Number of outliers detected: {len(outliers_mod)}")
print(f"\nOutlier statistics:")
print(outliers_mod.describe())

In [ ]:
# Visualize modified z-scores
plot_zscore(transformed_mod['m_zscore'], d=2.5, title='Height Modified Z-Scores (threshold=2.5)')

In [ ]:
# Compare distribution of modified z-scores
transformed_mod['m_zscore'].hist(bins=50, edgecolor='black')
plt.axvline(x=2.5, color='r', linestyle='--', label='Threshold=2.5')
plt.axvline(x=-2.5, color='r', linestyle='--')
plt.xlabel('Modified Z-score')
plt.ylabel('Frequency')
plt.title('Distribution of Modified Z-Scores')
plt.legend()
plt.show()

### Exercise 4.1: Compare Standard vs Modified Z-Score

In [ ]:
# Compare standard z-score (threshold=3) vs modified z-score (threshold=2.5)
outliers_standard, _ = zscore(wh['Height'], threshold=3)
outliers_modified, _ = modified_zscore(wh['Height'], threshold=2.5)

print("Comparison of Methods:")
print(f"Standard Z-score (t=3):  {len(outliers_standard)} outliers")
print(f"Modified Z-score (t=2.5): {len(outliers_modified)} outliers")

# Check overlap
overlap = set(outliers_standard.index) & set(outliers_modified.index)
print(f"\nOverlapping outliers: {len(overlap)}")

**Question:** Why might the Modified Z-score detect different outliers than standard Z-score?

**Your answer:** _[Your answer here]_

---

# Recipe 5: Multivariate Outlier Detection (NEW)

## Learning Objectives
- Understand why univariate methods fail on multivariate outliers
- Implement Mahalanobis Distance for multivariate outlier detection
- Apply Isolation Forest for anomaly detection
- Compare multivariate vs univariate approaches

## Key Concepts

**The Problem:** A data point can be an outlier in the multivariate space even if each individual feature is normal!

Example: A person who is 5'2" and weighs 200 lbs:
- Height alone: Not necessarily an outlier
- Weight alone: Not necessarily an outlier
- **Combination**: Definitely unusual!

### Two Approaches:

1. **Mahalanobis Distance**: Measures distance from center accounting for correlations
2. **Isolation Forest**: Uses decision trees to isolate anomalies

## 5.1 Visualize the Multivariate Distribution

In [ ]:
# Create a scatter plot to see the relationship between Height and Weight
plt.figure(figsize=(10, 6))
plt.scatter(wh['Height'], wh['Weight'], alpha=0.5, s=10)
plt.xlabel('Height (inches)')
plt.ylabel('Weight (pounds)')
plt.title('Height vs Weight Distribution')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Correlation between Height and Weight: {wh['Height'].corr(wh['Weight']):.3f}")

## 5.2 Understanding Mahalanobis Distance

**Mahalanobis Distance** accounts for correlations between variables:

$$D_M(x) = \sqrt{(x - \mu)^T \Sigma^{-1} (x - \mu)}$$

Where:
- x = data point
- μ = mean vector
- Σ = covariance matrix

**Intuition:** It measures how many "standard deviations" away a point is, considering correlations.

## 🎯 Your Task: Implement Mahalanobis Distance

Complete the `detect_mahalanobis_outliers` function:

In [ ]:
def detect_mahalanobis_outliers(df, threshold=3):
    """
    Detect outliers using Mahalanobis Distance.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Data with multiple numeric columns
    threshold : float, default=3
        Threshold for Mahalanobis distance (chi-square based)
    
    Returns:
    --------
    tuple: (outliers, distances)
        - outliers: DataFrame of outlier points
        - distances: Series of Mahalanobis distances for all points
    """
    # TODO: Calculate mean vector of the data
    # Hint: Use df.mean()
    mean = # YOUR CODE HERE
    
    # TODO: Calculate covariance matrix
    # Hint: Use df.cov()
    cov_matrix = # YOUR CODE HERE
    
    # TODO: Calculate inverse of covariance matrix
    # Hint: Use np.linalg.inv()
    inv_cov = # YOUR CODE HERE
    
    # Calculate Mahalanobis distance for each point
    distances = []
    for idx in df.index:
        # TODO: Get the data point as array
        point = # YOUR CODE HERE
        
        # TODO: Calculate Mahalanobis distance using scipy's mahalanobis function
        # Hint: mahalanobis(point, mean, inv_cov)
        dist = # YOUR CODE HERE
        distances.append(dist)
    
    # Create series of distances
    distances = pd.Series(distances, index=df.index, name='mahalanobis_distance')
    
    # Determine threshold based on chi-square distribution
    # For 2 degrees of freedom (2 variables), threshold^2 follows chi-square
    chi2_threshold = stats.chi2.ppf(0.95, df=df.shape[1])
    distance_threshold = np.sqrt(chi2_threshold)
    
    print(f"Using distance threshold: {distance_threshold:.2f} (95th percentile)")
    
    # TODO: Identify outliers where distance > threshold
    outlier_mask = # YOUR CODE HERE
    outliers = df[outlier_mask].copy()
    outliers['mahalanobis_distance'] = distances[outlier_mask]
    
    return outliers, distances

## Test Mahalanobis Distance

In [ ]:
# Apply Mahalanobis distance to Height and Weight
outliers_mahal, distances_mahal = detect_mahalanobis_outliers(wh[['Height', 'Weight']])

print(f"Number of outliers detected: {len(outliers_mahal)}")
print(f"\nTop 5 outliers by Mahalanobis distance:")
print(outliers_mahal.nlargest(5, 'mahalanobis_distance'))

In [ ]:
# Visualize Mahalanobis distances
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: Scatter plot with outliers highlighted
axes[0].scatter(wh['Height'], wh['Weight'], c='blue', alpha=0.3, s=10, label='Normal')
axes[0].scatter(outliers_mahal['Height'], outliers_mahal['Weight'], 
                c='red', s=50, label='Outliers', edgecolors='black', linewidth=1)
axes[0].set_xlabel('Height (inches)')
axes[0].set_ylabel('Weight (pounds)')
axes[0].set_title('Mahalanobis Outliers in 2D Space')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: Distribution of distances
axes[1].hist(distances_mahal, bins=50, edgecolor='black')
chi2_threshold = np.sqrt(stats.chi2.ppf(0.95, df=2))
axes[1].axvline(x=chi2_threshold, color='r', linestyle='--', 
                label=f'Threshold={chi2_threshold:.2f}', linewidth=2)
axes[1].set_xlabel('Mahalanobis Distance')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Mahalanobis Distances')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5.3 Isolation Forest

**Isolation Forest** works differently:
- Builds random decision trees
- Anomalies are easier to "isolate" (require fewer splits)
- Returns an anomaly score: -1 for outliers, 1 for inliers

**Advantages:**
- No assumptions about data distribution
- Scales well to high dimensions
- Can handle complex, non-linear relationships

## 🎯 Your Task: Implement Isolation Forest Detection

In [ ]:
def detect_isolation_forest_outliers(df, contamination=0.05, random_state=42):
    """
    Detect outliers using Isolation Forest.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Data with numeric columns
    contamination : float, default=0.05
        Expected proportion of outliers (0.01 to 0.5)
    random_state : int, default=42
        Random seed for reproducibility
    
    Returns:
    --------
    tuple: (outliers, predictions, scores)
        - outliers: DataFrame of detected outliers
        - predictions: Array of predictions (-1 for outliers, 1 for inliers)
        - scores: Array of anomaly scores (lower = more anomalous)
    """
    # TODO: Create an Isolation Forest model
    # Hint: IsolationForest(contamination=..., random_state=...)
    iso_forest = # YOUR CODE HERE
    
    # TODO: Fit the model and predict
    # Hint: Use fit_predict(df)
    predictions = # YOUR CODE HERE
    
    # TODO: Get anomaly scores
    # Hint: Use score_samples(df)
    scores = # YOUR CODE HERE
    
    # TODO: Create DataFrame of outliers (where prediction == -1)
    outlier_mask = # YOUR CODE HERE
    outliers = df[outlier_mask].copy()
    outliers['anomaly_score'] = scores[outlier_mask]
    
    return outliers, predictions, scores

## Test Isolation Forest

In [ ]:
# Apply Isolation Forest
outliers_iso, predictions_iso, scores_iso = detect_isolation_forest_outliers(
    wh[['Height', 'Weight']], 
    contamination=0.05
)

print(f"Number of outliers detected: {len(outliers_iso)}")
print(f"Percentage of data: {len(outliers_iso)/len(wh)*100:.2f}%")
print(f"\nTop 5 outliers by anomaly score (most anomalous):")
print(outliers_iso.nsmallest(5, 'anomaly_score'))

In [ ]:
# Visualize Isolation Forest results
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: Scatter plot with outliers highlighted
normal_data = wh[predictions_iso == 1]
axes[0].scatter(normal_data['Height'], normal_data['Weight'], 
                c='blue', alpha=0.3, s=10, label='Normal')
axes[0].scatter(outliers_iso['Height'], outliers_iso['Weight'], 
                c='red', s=50, label='Outliers', edgecolors='black', linewidth=1)
axes[0].set_xlabel('Height (inches)')
axes[0].set_ylabel('Weight (pounds)')
axes[0].set_title('Isolation Forest Outliers in 2D Space')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: Distribution of anomaly scores
axes[1].hist(scores_iso, bins=50, edgecolor='black')
axes[1].axvline(x=outliers_iso['anomaly_score'].max(), color='r', 
                linestyle='--', label='Threshold', linewidth=2)
axes[1].set_xlabel('Anomaly Score (lower = more anomalous)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Anomaly Scores')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5.4 Compare All Multivariate Methods

In [ ]:
# Compare Mahalanobis vs Isolation Forest
print("Method Comparison:")
print(f"Mahalanobis Distance: {len(outliers_mahal)} outliers")
print(f"Isolation Forest:     {len(outliers_iso)} outliers")

# Check overlap
overlap = set(outliers_mahal.index) & set(outliers_iso.index)
print(f"\nOverlapping outliers: {len(overlap)}")
print(f"Agreement: {len(overlap)/max(len(outliers_mahal), len(outliers_iso))*100:.1f}%")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# All data
axes[0].scatter(wh['Height'], wh['Weight'], c='lightgray', alpha=0.5, s=10)
axes[0].set_xlabel('Height (inches)')
axes[0].set_ylabel('Weight (pounds)')
axes[0].set_title('All Data')
axes[0].grid(True, alpha=0.3)

# Mahalanobis outliers
axes[1].scatter(wh['Height'], wh['Weight'], c='lightgray', alpha=0.3, s=10)
axes[1].scatter(outliers_mahal['Height'], outliers_mahal['Weight'], 
                c='red', s=50, label='Mahalanobis', edgecolors='black', linewidth=1)
axes[1].set_xlabel('Height (inches)')
axes[1].set_ylabel('Weight (pounds)')
axes[1].set_title(f'Mahalanobis Distance ({len(outliers_mahal)} outliers)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Isolation Forest outliers
axes[2].scatter(wh['Height'], wh['Weight'], c='lightgray', alpha=0.3, s=10)
axes[2].scatter(outliers_iso['Height'], outliers_iso['Weight'], 
                c='blue', s=50, label='Isolation Forest', edgecolors='black', linewidth=1)
axes[2].set_xlabel('Height (inches)')
axes[2].set_ylabel('Weight (pounds)')
axes[2].set_title(f'Isolation Forest ({len(outliers_iso)} outliers)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Exercise 5.1: Why Multivariate Matters

Let's demonstrate why univariate methods miss multivariate outliers:

In [ ]:
# Get univariate outliers
height_outliers_z, _ = zscore(wh['Height'], threshold=3)
weight_outliers_z, _ = zscore(wh['Weight'], threshold=3)
univariate_outliers = set(height_outliers_z.index) | set(weight_outliers_z.index)

# Get multivariate outliers
multivariate_outliers = set(outliers_mahal.index)

# Find outliers ONLY detected by multivariate method
only_multivariate = multivariate_outliers - univariate_outliers

print(f"Univariate outliers (Z-score):    {len(univariate_outliers)}")
print(f"Multivariate outliers (Mahalanobis): {len(multivariate_outliers)}")
print(f"\nOutliers ONLY found by multivariate method: {len(only_multivariate)}")
print(f"\nExample points missed by univariate methods:")
print(wh.loc[list(only_multivariate)][:5])

**Question:** Why do these points appear normal in univariate analysis but unusual in multivariate analysis?

**Your answer:** _[Your answer here]_

### Exercise 5.2: Contamination Sensitivity

Test how the contamination parameter affects Isolation Forest:

In [ ]:
# TODO: Test contamination values: 0.01, 0.05, 0.10
contamination_values = [0.01, 0.05, 0.10]

print("Contamination Sensitivity Analysis:")
for c in contamination_values:
    outliers, _, _ = detect_isolation_forest_outliers(wh[['Height', 'Weight']], contamination=c)
    print(f"Contamination={c}: {len(outliers)} outliers ({len(outliers)/len(wh)*100:.2f}%)")

---

# Reflection Questions

Answer these questions based on your experience with the activity:

## 1. Method Selection
**Question:** When would you choose each method?
- Tukey's IQR: _[Your answer]_
- Z-score: _[Your answer]_
- Modified Z-score: _[Your answer]_
- Mahalanobis Distance: _[Your answer]_
- Isolation Forest: _[Your answer]_

## 2. Assumptions and Limitations
**Question:** What assumptions does each method make?
- IQR: _[Your answer]_
- Z-score: _[Your answer]_
- Modified Z-score: _[Your answer]_
- Mahalanobis: _[Your answer]_
- Isolation Forest: _[Your answer]_

## 3. Practical Considerations
**Question:** What factors should you consider when choosing a threshold?

_[Your answer]_

## 4. Multivariate vs Univariate
**Question:** Why is it important to use multivariate methods when analyzing datasets with multiple features?

_[Your answer]_

---

# Challenge Exercise: Apply to New Dataset

Now apply what you've learned to a synthetic dataset with known outliers!

## Generate Challenge Dataset

In [ ]:
# Generate synthetic data with known outliers
np.random.seed(42)

# Normal data
n_samples = 1000
mean = [50, 100]
cov = [[10, 8], [8, 20]]  # Correlated variables
normal_data = np.random.multivariate_normal(mean, cov, n_samples)

# Add outliers
n_outliers = 50
outlier_data = np.random.uniform([30, 50], [70, 150], (n_outliers, 2))

# Combine
all_data = np.vstack([normal_data, outlier_data])
labels = np.array([0] * n_samples + [1] * n_outliers)  # 0=normal, 1=outlier

# Create DataFrame
challenge_df = pd.DataFrame(all_data, columns=['Feature_1', 'Feature_2'])
challenge_df['true_label'] = labels

print(f"Challenge dataset shape: {challenge_df.shape}")
print(f"True outliers: {n_outliers} ({n_outliers/len(challenge_df)*100:.1f}%)")

In [ ]:
# Visualize the challenge dataset
plt.figure(figsize=(10, 6))
colors = ['blue' if label == 0 else 'red' for label in challenge_df['true_label']]
plt.scatter(challenge_df['Feature_1'], challenge_df['Feature_2'], 
            c=colors, alpha=0.5, s=20)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Challenge Dataset (Red = True Outliers)')
plt.grid(True, alpha=0.3)
plt.show()

## 🎯 Your Challenge Tasks:

1. Apply at least 3 different outlier detection methods to this dataset
2. Compare their performance against the true labels
3. Calculate metrics: precision, recall, F1-score
4. Visualize the results
5. Determine which method works best and explain why

In [ ]:
# Helper function to calculate metrics
def calculate_metrics(true_labels, predicted_outliers_idx):
    """
    Calculate precision, recall, and F1-score.
    
    Parameters:
    -----------
    true_labels : array-like
        True labels (0=normal, 1=outlier)
    predicted_outliers_idx : list or set
        Indices of predicted outliers
    """
    # Create prediction array
    predicted = np.zeros(len(true_labels))
    predicted[list(predicted_outliers_idx)] = 1
    
    # Calculate metrics
    true_positives = np.sum((predicted == 1) & (true_labels == 1))
    false_positives = np.sum((predicted == 1) & (true_labels == 0))
    false_negatives = np.sum((predicted == 0) & (true_labels == 1))
    
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'true_positives': true_positives,
        'false_positives': false_positives,
        'false_negatives': false_negatives
    }

## TODO: Apply Method 1 - Your Choice

In [ ]:
# TODO: Apply your first method
# method_1_outliers = ...
# metrics_1 = calculate_metrics(challenge_df['true_label'], method_1_outliers.index)
# print("Method 1 Results:")
# print(f"Precision: {metrics_1['precision']:.3f}")
# print(f"Recall: {metrics_1['recall']:.3f}")
# print(f"F1-Score: {metrics_1['f1_score']:.3f}")

## TODO: Apply Method 2 - Your Choice

In [ ]:
# TODO: Apply your second method
# method_2_outliers = ...
# metrics_2 = calculate_metrics(challenge_df['true_label'], method_2_outliers.index)
# print("Method 2 Results:")
# print(f"Precision: {metrics_2['precision']:.3f}")
# print(f"Recall: {metrics_2['recall']:.3f}")
# print(f"F1-Score: {metrics_2['f1_score']:.3f}")

## TODO: Apply Method 3 - Your Choice

In [ ]:
# TODO: Apply your third method
# method_3_outliers = ...
# metrics_3 = calculate_metrics(challenge_df['true_label'], method_3_outliers.index)
# print("Method 3 Results:")
# print(f"Precision: {metrics_3['precision']:.3f}")
# print(f"Recall: {metrics_3['recall']:.3f}")
# print(f"F1-Score: {metrics_3['f1_score']:.3f}")

## TODO: Compare All Methods

In [ ]:
# TODO: Create a comparison visualization and summary table
# Show which method performed best and explain why

## Challenge Analysis

**Which method worked best?**

_[Your answer]_

**Why did it work better than the others?**

_[Your answer]_

**What trade-offs did you observe between precision and recall?**

_[Your answer]_

**In a real-world scenario, would you prioritize precision or recall for outlier detection? Why?**

_[Your answer]_

---

# Summary

Congratulations! You've completed the Foundational Statistical Outlier Detection activity.

## Key Takeaways:

1. **Visualization is essential** for understanding your data and identifying potential outliers
2. **Tukey's IQR method** is robust and doesn't assume normality
3. **Z-score methods** work well for normal distributions but can be sensitive to outliers
4. **Modified Z-score** is more robust than standard Z-score
5. **Multivariate methods** (Mahalanobis, Isolation Forest) are necessary when features are correlated
6. **No single method is perfect** - always compare multiple approaches
7. **Domain knowledge** is crucial for interpreting results and setting thresholds

## Next Steps:

- Explore time-series outlier detection methods
- Learn about ensemble methods combining multiple approaches
- Study deep learning approaches for anomaly detection
- Apply these methods to your own datasets!